# Exploratory Data Analysis

## Configuration

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append("../utils")
from download_parquet import download_dataset
import numpy as np
import json

    

DOWNLOAD_DATA = False
script_path = Path('..') / 'utils' / '00_donwload_parquet.py'
parquet_path = Path('..') / 'data' / 'clinical_trials.parquet'

## Download Hugging Face Data (Optativo)

In [ ]:

if DOWNLOAD_DATA:

    download_dataset()
else:
    print('No se descargaron los datos. Si deseas descargar los datos, establece DOWNLOAD_DATA en True.')

No se descargaron los datos. Si deseas descargar los datos, establece DOWNLOAD_DATA en True.


## Load Dataframe

In [ ]:
df = pd.read_parquet(parquet_path)
df.head()

,nct_id,brief_title,official_title,overall_status,completion_date,brief_summary,conditions,keywords,phases,arm_groups,eligibility_criteria,min_age,std_ages,locations,condition_meshes,has_results
0,NCT05155956,Cerebral Hemodynamics and Microemboli During P...,Cerebral Hemodynamics and Microemboli During P...,ACTIVE_NOT_RECRUITING,2026-12,This is an observational cohort study addressi...,"[""Thoracic Aortic Aneurysm"", ""Thoracic Aortic ...",[],[],[],Inclusion Criteria:\n\n* Enrollment in NCT0321...,18 Years,"[""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""19104"", ""country"": ""...","[{""id"": ""D017545"", ""term"": ""Aortic Aneurysm, T...",False
1,NCT06558656,Effect of Food of NTQ5082 in Healthy Subjects,"A Randomized, Open Label, Two Sequence, Two Pe...",NOT_YET_RECRUITING,2024-12,NTQ5082 capsule is a small molecule CFB factor...,"[""Complement-mediated Hemolytic Diseases""]",[],"[""PHASE1""]","[{""label"": ""Fasting-Fed"", ""description"": ""Fast...",Inclusion Criteria:\n\n1. Healthy male or fema...,18 Years,"[""ADULT""]","[{""status"": null, ""zip"": ""410035"", ""country"": ...",[],False
2,NCT04613856,Water Bolus Volumes During Continuous Exercise...,Water Bolus Volumes During Continuous Exercise...,COMPLETED,2020-03-12,Hydration is important to all individuals incl...,"[""Dehydration (Physiology)"", ""Hyperthermia""]",[],"[""NA""]","[{""label"": ""Current Best Practice - 237 mL wat...",Inclusion Criteria:\n\n* Male\n* 18-39 y old\n...,18 Years,"[""ADULT""]","[{""status"": null, ""zip"": ""14214"", ""country"": ""...","[{""id"": ""D003681"", ""term"": ""Dehydration""}, {""i...",False
3,NCT04849156,Spermatozoa Morphology Selection by Thermotaxis,Sperm Selection by Thermotaxis: a Novel Techni...,COMPLETED,2022-01-05,This study aims to eventually assess the usefu...,"[""Male Infertility""]","[""Spermatozoa selection"", ""Thermotaxis"", ""Test...",[],[],Inclusion Criteria:\n\n* Age: ≥18 and \<40 yea...,18 Years,"[""ADULT""]","[{""status"": null, ""zip"": null, ""country"": ""Por...","[{""id"": ""D007248"", ""term"": ""Infertility, Male""...",False
4,NCT04530656,Phase I Trial of a Recombinant SARS-CoV-2 Vacc...,"Safety, Tolerability and Immunogenicity of a R...",COMPLETED,2020-10-28,"This is a phase I, single-center, randomized, ...","[""COVID-19""]","[""Safety"", ""Tolerability"", ""Immunogenicity"", ""...","[""PHASE1""]","[{""label"": ""Middle-dose vaccine (18-55 years) ...",Inclusion Criteria:\n\n* Male or female subjec...,18 Years,"[""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""210009"", ""country"": ...","[{""id"": ""D000086382"", ""term"": ""COVID-19""}]",False


## Análisis de Nulos

Para columnas escalares se detecta `NaN`/`None`. Para columnas que contienen arrays (listas), un valor "nulo" equivale a una lista vacía `[]`.

In [ ]:

# ── helpers ──────────────────────────────────────────────────────────────────

def _parse_if_json_list(x):
    """Intenta convertir x a lista Python. Retorna la lista (puede estar vacía) o None."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list):
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass
    return None  # escalar no-lista

def _is_array_col(series):
    """True si la columna contiene arrays (listas Python, ndarray o strings JSON de lista)."""
    for val in series.dropna().head(500):
        if _parse_if_json_list(val) is not None:   # [] vacío también es válido
            return True
    return False

def _is_empty_array(x):
    """True si el valor representa un array vacío ([], np.array([]), '[]', None excluido)."""
    parsed = _parse_if_json_list(x)
    return isinstance(parsed, list) and len(parsed) == 0

# ── clasificar columnas ──────────────────────────────────────────────────────

array_cols  = [col for col in df.columns if _is_array_col(df[col])]
scalar_cols = [col for col in df.columns if col not in array_cols]

print("Columnas array  :", array_cols)
print("Columnas escalar:", scalar_cols)

# ── nulos escalares (NaN / None) ─────────────────────────────────────────────

scalar_nulls     = df[scalar_cols].isnull().sum().rename("nulos")
scalar_nulls_pct = (df[scalar_cols].isnull().mean() * 100).rename("pct_nulos")

# ── "nulos" en arrays (lista vacía) ─────────────────────────────────────────

array_empty     = pd.Series(
    {col: df[col].apply(_is_empty_array).sum()        for col in array_cols},
    name="nulos"
)
array_empty_pct = pd.Series(
    {col: df[col].apply(_is_empty_array).mean() * 100 for col in array_cols},
    name="pct_nulos"
)

# ── reporte unificado ────────────────────────────────────────────────────────

null_report = pd.concat([
    pd.concat([scalar_nulls, scalar_nulls_pct], axis=1).assign(tipo_columna="escalar"),
    pd.concat([array_empty,  array_empty_pct],  axis=1).assign(tipo_columna="array"),
])

null_report["pct_nulos"] = null_report["pct_nulos"].round(2)
null_report = null_report.sort_values("pct_nulos", ascending=False)

print(f"\nTotal de registros: {len(df):,}\n")
null_report

Columnas array  : ['conditions', 'keywords', 'phases', 'arm_groups', 'std_ages', 'locations', 'condition_meshes']
Columnas escalar: ['nct_id', 'brief_title', 'official_title', 'overall_status', 'completion_date', 'brief_summary', 'eligibility_criteria', 'min_age', 'has_results']

Total de registros: 440,579



,nulos,pct_nulos,tipo_columna
keywords,157281,35.70,array
phases,101086,22.94,array
condition_meshes,93531,21.23,array
arm_groups,51101,11.60,array
locations,36630,8.31,array
min_age,28012,6.36,escalar
completion_date,14110,3.20,escalar
official_title,6852,1.56,escalar
eligibility_criteria,39,0.01,escalar
nct_id,0,0.00,escalar
